# MiniDance — SFT 精调（VS Code + Colab 运行时）

在预训练（`runs/s4_base/ckpt.pt`）的基础上，用高质量片段 + 短字幕精调。

**连接方法**：右上角「选择内核」→ Colab → 新建 Colab 服务器 → 选 **L4**。

从上到下依次运行全部单元格即可：
1. 挂载网盘、拉取最新代码（包含 `trainer/train_sft.py`）
2. 安装依赖
3. 把 latent 数据从网盘复制到 Colab 本地磁盘
4. SFT 训练：约 1.1 万条高美学片段，3000 步，L4 约 30 分钟
5. 查看样例 GIF 和日志

结果保存在网盘 `MyDrive/minidance/runs/sft_base/`，断开后重新运行全部单元格会接着训。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. 挂载网盘，拉取代码


In [ ]:
REPO_URL = 'https://github.com/JuniverseLi/minidance.git'
DRIVE_DIR = '/content/drive/MyDrive/minidance'

import os
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        raise RuntimeError(f'网盘挂载失败（{type(e).__name__}: {e}）。'
                           '如果 VS Code 里挂载不了，请改用 Colab 网页版运行 notebooks/train_colab.ipynb') from e
assert os.path.exists(f'{DRIVE_DIR}/latents'), f'网盘里没找到 {DRIVE_DIR}/latents'
print('网盘已挂载:', os.listdir(DRIVE_DIR))

if not os.path.exists('/content/minidance'):
    !git clone -q {REPO_URL} /content/minidance
else:
    !git -C /content/minidance fetch -q origin
    !git -C /content/minidance reset -q --hard origin/main
%cd /content/minidance
!git log --oneline -1

In [ ]:
!pip install -q diffusers safetensors "transformers>=4.51"
import transformers; print('transformers', transformers.__version__)

## 2. 把数据从网盘复制到 Colab 本地磁盘

已经复制过的文件会跳过。


In [ ]:
import glob, time
SRC, DST = f'{DRIVE_DIR}/latents', '/content/latents'
os.makedirs(DST, exist_ok=True)
print(len(glob.glob(f'{SRC}/shard_*.safetensors')), 'shards on Drive')
t = time.time()
!cp -n {SRC}/shard_* {DST}/
n = len(glob.glob(f'{DST}/shard_*.safetensors'))
size = sum(os.path.getsize(p) for p in glob.glob(f'{DST}/shard_*')) / 2**30
print(f'{n} shards, {size:.1f} GB local, copy took {time.time() - t:.0f}s')

CAPS = f'{DRIVE_DIR}/short_captions.jsonl'
if os.path.exists(CAPS):
    print('短字幕:', sum(1 for _ in open(CAPS, encoding='utf-8')), '条')
else:
    print('未找到短字幕文件，只用长字幕训练')
    CAPS = 'none'

## 3. SFT 训练

- 从预训练的 EMA 权重出发，只用美学分 ≥ 6.0、有运动的片段，只用短字幕（中英文），学习率 5e-5
- 星空、雾这类少见场景会多采样一些
- 每 500 步出一张样例 GIF、存一次 checkpoint
- 日志开头应出现 `select(...) -> 11,xxx clips` 和 `initialized from ... (step 64000)`


In [ ]:
SFT_RUN = 'sft_base'
INIT = f'{DRIVE_DIR}/runs/s4_base/ckpt.pt'   # 预训练的最终权重（不会被修改）
SFT_OUT = f'{DRIVE_DIR}/runs/{SFT_RUN}'
STEPS = 3000
BATCH = 32
MINUTES = 90
assert os.path.exists(INIT), f'没找到预训练权重 {INIT}'
assert CAPS != 'none', 'SFT 需要短字幕文件 short_captions.jsonl'

!python trainer/train_sft.py --data_dir /content/latents --short_captions {CAPS}     --init {INIT} --resume --preset base --batch {BATCH} --steps {STEPS}     --max_minutes {MINUTES} --compile --out {SFT_OUT}


## 4. 查看样例和日志


In [ ]:
from IPython.display import Image, display
gifs = sorted(glob.glob(f'{SFT_OUT}/samples/*.gif'))
print(len(gifs), 'SFT sample GIFs；每个 GIF 从左到右：4 条固定提示词 + 2 条训练集短字幕')
for g in gifs[-2:]:
    print(os.path.basename(g))
    display(Image(filename=g))


In [ ]:
!grep "initialized\|resumed\|DiT-\|Traceback\|Error" {SFT_OUT}/train.log | cut -c1-160 | head -5
print('...')
!grep "step \|reached\|samples ->" {SFT_OUT}/train.log | cut -c1-160 | tail -8
